In [ ]:
"""
Script: train_knots_3d.ipynb

Description:
    Loads 3D knot singularity CSV data from multiple Rytov variance experiments,
    constructs a PyTorch Dataset and DataLoader, initializes a 3D convolutional
    classifier (Classifier3D), trains it with learning rate scheduling,
    and saves the final model checkpoint.

Sections:
    1. Imports & Device Setup
    2. Configuration: resolution, hyperparameters, model architecture
    3. Dataset Loading & Preprocessing
    4. DataLoader Creation
    5. Model Initialization & Summary
    6. Training Utilities: loops & plotting
    7. Training Loop
    8. Model Saving
"""

In [ ]:
# -----------------------------------------------------------------------------
# 1. Imports & Device Setup
# -----------------------------------------------------------------------------
import sys
sys.path.append('../')  # Add parent directory for project modules

import time                    # For tracking training duration
import json, csv               # For reading/writing experiment data
from extra_functions_package.all_knots_functions import *  # Knot utilities
from torch.utils.data import TensorDataset, DataLoader       # Dataset handling
from torch.optim.lr_scheduler import ReduceLROnPlateau      # LR scheduling
import torch                                              # Core PyTorch
import torch.nn as nn                                     # Neural network modules
import torch.nn.functional as F                           # Activation & loss functions
from tqdm import trange                                    # Progress bars
from torchsummary import summary                          # Model summary
from classifier_models import Classifier3D                 # 3D CNN model definition

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# -----------------------------------------------------------------------------
# 2. Configuration: resolution, hyperparameters, model architecture
# -----------------------------------------------------------------------------
desired_res = (32, 32, 32)  # Target input shape: (depth, height, width)

# Training hyperparameters
hyperparams = {
    'learning_rate': 1e-5,   # Initial learning rate
    'patience': 0,           # LR scheduler patience (unused until decay_epoch)
    'decay_epoch': 25,       # Epoch to apply LR step
    'factor': 0.2,           # Multiplicative LR decay factor
    'batch_size': 64         # Mini-batch size
}
num_epochs = 50            # Total number of training epochs
print_every = 1            # Print progress every N epochs

# Define convolutional feature stages
stages = [
    [(1, 32, 3, 1, 1), (32, 32, 3, 1, 1), (32, 32, 3, 1, 1)],
    [(32, 64, 5, 1, 1), (64, 64, 5, 1, 1), (64, 64, 5, 1, 1)]
]
# Pooling after each stage (kernel_size, stride, padding)
pooling_configs = [(2, 2, 1), (2, 2, 1)]

# -----------------------------------------------------------------------------
# 3. Dataset Loading & Preprocessing
# -----------------------------------------------------------------------------
# Define knot classes and data folders
knot_types = {
    'standard_14': 0, 'standard_16': 1, 'standard_18': 2,
    '30both': 3, '30oneZ': 4, 'optimized': 5, 'pm_03_z': 6,
    '30oneX': 7, '15oneZ': 8, 'trefoil_standard_12': 9,
    'trefoil_optimized': 10
}
knots = list(knot_types.keys())
folders = [
    '../HOPFS_L270_0.05_1000_64x64x64_v1',
    '../HOPFS_L270_0.15_1000_64x64x64_v1',
    '../HOPFS_L270_0.25_1000_64x64x64_v1',
]

num_classes = len(knots)
X_list, Y_list = [], []
csv.field_size_limit(10_000_000)  # Allow large JSON entries
flag_print_shape = True           # Print input shape once for verification

# Iterate over folders and load point coordinates from CSV
for folder in folders:
    for knot in knots:
        filename = f"{folder}/data_{knot}.csv"
        try:
            with open(filename, 'r') as file:
                reader = csv.reader(file)
                for row in reader:
                    # Each row is JSON: [idx, (Nx,Ny,Nz), points...]
                    data = json.loads(row[0])
                    data_array = np.array(data, dtype=object)

                    # Extract grid dimensions and coordinate list
                    Nx, Ny, Nz = data_array[1]
                    points_list = np.array(data_array[2:], dtype=int)

                    # Print shape once
                    if flag_print_shape:
                        print(f"Grid shape: {Nx}x{Ny}x{Nz}")
                        flag_print_shape = False

                    # Rescale points to match desired_res if needed
                    if (Nx, Ny, Nz) != desired_res:
                        scales = np.array(desired_res) / np.array([Nx, Ny, Nz])
                        points_list = np.rint(points_list * scales).astype(int)

                    # Build binary 3D voxel grid
                    voxels = np.zeros(desired_res, dtype=int)
                    for x, y, z in points_list:
                        if 0 <= x < desired_res[0] and 0 <= y < desired_res[1] and 0 <= z < desired_res[2]:
                            voxels[x, y, z] = 1

                    X_list.append(voxels)
                    Y_list.append(knot_types[knot])
        except FileNotFoundError:
            print(f"Missing file: {filename}")
        except json.JSONDecodeError:
            print(f"JSON decode error: {filename}")

print(f"Loaded total samples: {len(X_list)} ({len(X_list)//num_classes} per class)")


In [ ]:
# -----------------------------------------------------------------------------
# 4. DataLoader Creation
# -----------------------------------------------------------------------------
# Convert lists to tensors and one-hot encode labels
X_np = np.stack(X_list)  # Shape: [N, D, H, W]
y_np = np.array(Y_list)
X_tensor = torch.tensor(X_np).unsqueeze(1).float()  # [N,1,D,H,W]
y_tensor = F.one_hot(torch.tensor(y_np), num_classes).float()

dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=hyperparams['batch_size'], shuffle=True)

In [ ]:
# -----------------------------------------------------------------------------
# 5. Model Initialization & Summary
# -----------------------------------------------------------------------------
model = Classifier3D(stages, pooling_configs, num_classes=num_classes).to(device)
model.initialize_weights()  # Custom weight initialization
summary(model, input_size=X_tensor.shape[1:])  # Print architecture details

In [ ]:
# -----------------------------------------------------------------------------
# 6. Training Utilities: loops & plotting
# -----------------------------------------------------------------------------
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=hyperparams['learning_rate'])
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=hyperparams['factor'],
                              patience=hyperparams['patience'], verbose=True)

def loop_train(model, loader):
    """Train for one epoch and return average loss."""
    model.train()
    total_loss = 0.0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, torch.argmax(targets, dim=1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def plot_losses(losses):
    """Plot training loss curve."""
    plt.figure()
    plt.plot(losses, label='Train Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 7. Training Loop
# -----------------------------------------------------------------------------
train_losses = []
start_time = time.time()
scheduler.step(float('inf'))  # Initialize scheduler
for epoch in trange(num_epochs, desc='Training Epochs'):
    loss = loop_train(model, train_loader)
    train_losses.append(loss)
    # Apply LR decay at specified epoch
    if epoch == hyperparams['decay_epoch'] - 1:
        scheduler.step(loss)
    if (epoch + 1) % print_every == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss:.4f}")

total_time = time.time() - start_time
print(f"Total training time: {total_time:.1f}s")
plot_losses(train_losses)


In [ ]:
# -----------------------------------------------------------------------------
# 8. Model Saving
# -----------------------------------------------------------------------------
checkpoint = {
    'model_state_dict': model.state_dict(),
    'hyperparams': hyperparams,
    'num_classes': num_classes,
    'stages': stages,
    'pooling_configs': pooling_configs,
    'desired_res': desired_res,
}
torch.save(checkpoint, 'classifier_knots_3d_32_full_2.pth')
print("Saved model to classifier_knots_3d_32_full_2.pth")
